In [1]:
# packages
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
import json
import gc

In [2]:
# read the data
embeddings_path = Path("/kaggle/input/datasets/lianestrauch/scibert-base/bert-base/chunks")


chunk_files = sorted((embeddings_path).glob("chunk_*.parquet"))
print(len(chunk_files))

bert_embeddings = pd.read_parquet(
    chunk_files,
    columns=["id", "layer_2nd_last"],
)

########### GARBAGE COLLECTION ############
gc.collect()

125


0

In [3]:
# set model
model = "scibert"

# set layer
layer = "layer_11"

# set column name?

# set eps
eps_values = {
    100: np.round(np.arange(0.07, 0.09 + 0.01, 0.01), 2),
}

# set outputfile
output_file = Path(f"full_clustering_results_{model}_{layer}.json")


In [4]:
!pip install kDBCV

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 824.7 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 35.4 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.29.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cesium 0.12.4 

In [5]:
import json
import time
from pathlib import Path
import numpy as np

# Patch NumPy 2.0 compatibility for legacy libraries like kDBCV
if not hasattr(np, 'float_'):
    np.float_ = np.float64
if not hasattr(np, 'int_'):
    np.int_ = np.int64

from kDBCV import DBCV_score
from scipy.spatial.distance import cosine
from sklearn.preprocessing import normalize

from sklearn.cluster import DBSCAN

# ============================================================
# Load existing results, or create a new dictionary
# ============================================================

if output_file.exists():
    with open(output_file, "r") as f:
        clustering_results = json.load(f)

    print(
        f"Loaded {len(clustering_results)} existing clustering results."
    )
else:
    clustering_results = {}

    print("No existing results found. Starting a new file.")


# ============================================================
# Prepare embeddings
# ============================================================

col_name = bert_embeddings.columns[-1]

embeddings = np.vstack(bert_embeddings[col_name].values)
n_samples = len(embeddings)

print(f"Number of data points: {n_samples}")


# ============================================================
# Run clustering
# ============================================================

for min_samples, current_eps_values in eps_values.items():

    for eps in current_eps_values:

        key = f"eps_{eps:.4f}_minPts_{min_samples}"

        # ----------------------------------------------------
        # Skip if this combination has already been calculated
        # ----------------------------------------------------
        if key in clustering_results:
            print(
                f"SKIPPING: eps={eps:.4f}, "
                f"minPts={min_samples} "
                f"(already calculated)"
            )
            continue

        print(
            f"Running: eps={eps:.4f}, "
            f"minPts={min_samples}..."
        )

        start_time = time.perf_counter()

        labels = DBSCAN(
            eps=eps,
            min_samples=min_samples,
            metric="cosine"
        ).fit_predict(embeddings)

        elapsed_time = time.perf_counter() - start_time

        # ----------------------------------------------------
        # Cluster statistics
        # ----------------------------------------------------

        unique_labels, counts = np.unique(
            labels,
            return_counts=True
        )

        # Exclude noise (-1)
        cluster_counts = counts[unique_labels != -1]

        n_clusters = len(cluster_counts)
        n_noise = int(np.sum(labels == -1))

        # Largest cluster
        if len(cluster_counts) > 0:
            largest_cluster_size = int(np.max(cluster_counts))
        else:
            largest_cluster_size = 0

        # Percentage of full dataset
        largest_cluster_pct = (
            100 * largest_cluster_size / n_samples
            if n_samples > 0 else 0
        )

        # DBCV
        embeddings_norm = normalize(embeddings, norm='l2', axis=1) # dbcv only takes euclidean distance
        score = DBCV_score(embeddings_norm, labels)
        print("DBCV Score (based on normalised embeddings and euclidean distance):", score)

        # ----------------------------------------------------
        # Store result
        # ----------------------------------------------------

        clustering_results[key] = {
            "model": model,
            "layer": layer,
            "eps": float(eps),
            "min_samples": int(min_samples),
            "n_clusters": int(n_clusters),
            "n_noise": n_noise,
            "largest_cluster_size": largest_cluster_size,
            "largest_cluster_pct": float(largest_cluster_pct),
            "runtime_seconds": float(elapsed_time),
            "n_samples": int(n_samples),
            "dbcv": score,
            "labels": { str(sample_id): int(label) for sample_id, label in zip(bert_embeddings["id"], labels) }
        }

        print(
            f"  finished: "
            f"{n_clusters} clusters, "
            f"largest={largest_cluster_size} "
            f"({largest_cluster_pct:.2f}%), "
            f"time={elapsed_time:.2f}s"
        )

        # ----------------------------------------------------
        # Save immediately after each clustering
        # ----------------------------------------------------
        #
        # This is useful for long-running experiments:
        # if the script crashes halfway through, everything
        # completed so far is already saved.
        #

        with open(output_file, "w") as f:
            json.dump(clustering_results, f)


# ============================================================
# Done
# ============================================================

print(
    f"\nDone. Total stored results: "
    f"{len(clustering_results)}"
)

No existing results found. Starting a new file.
Number of data points: 499195
Running: eps=0.0700, minPts=100...
memory cutoff reached
DBCV Score (based on normalised embeddings and euclidean distance): (-1, None)
  finished: 4 clusters, largest=330742 (66.26%), time=7851.31s
Running: eps=0.0800, minPts=100...
memory cutoff reached
DBCV Score (based on normalised embeddings and euclidean distance): (-1, None)
  finished: 2 clusters, largest=398775 (79.88%), time=7884.60s
Running: eps=0.0900, minPts=100...
Not enough clusters: must have at least two.
DBCV Score (based on normalised embeddings and euclidean distance): (-1, None)
  finished: 1 clusters, largest=440769 (88.30%), time=8001.57s

Done. Total stored results: 3
